# Traffic recovery: resource-aware MOIRAI probe
Any allocated CUDA GPU; no A100 name requirement. High host RAM recommended for saved-tensor CPU offload. No guarantee that small-VRAM GPUs can fit Traffic. Pinned source, joint 862 channels, context 512, horizon 720, official loss. No main experiment or budget confirmation.


In [ ]:
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

assert (3, 11) <= sys.version_info[:2] <= (3, 12)
subprocess.run(["nvidia-smi"], check=True)
print("Kernel:", sys.executable, sys.version)
COMMIT = input("Full published Traffic recovery commit SHA: ").strip()
assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT)
FAMILY = "moirai1"

In [ ]:
ROOT = Path("/content") / ("tsfm-pilot-" + COMMIT)
URL = "https://github.com/Han-Youseung/tsfm-zero-few-shot-crossover.git"
if not ROOT.exists():
    subprocess.run(["git", "clone", URL, str(ROOT)], check=True)
assert (
    subprocess.check_output(
        ["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True
    ).strip()
    == URL
)
assert not subprocess.check_output(
    ["git", "-C", str(ROOT), "status", "--porcelain", "--untracked-files=no"], text=True
).strip()
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", COMMIT], check=True)
os.chdir(ROOT)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip() == COMMIT

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
PERSIST = Path("/content/drive/MyDrive/tsfm-traffic-recovery")
OUT = PERSIST / COMMIT / FAMILY
OUT.mkdir(parents=True, exist_ok=True)
CACHE = PERSIST / "public-source-cache"
CACHE.mkdir(parents=True, exist_ok=True)
print("Persistent results and checkpoints:", OUT)
print("Stale running.lock: confirm old process is dead before manually removing only that lock.")

In [ ]:
ENV = Path("/content") / ("venv-pilot-" + FAMILY)
PY = ENV / "bin/python"
if not PY.exists():
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", f"python{version}-venv"], check=True)
    subprocess.run([sys.executable, "-m", "venv", str(ENV)], check=True)
subprocess.run(
    [str(PY), "-m", "pip", "install", "-r", "requirements/traffic-probe.txt"], check=True
)
subprocess.run([str(PY), "-m", "pip", "install", "-e", ".[dev]"], check=True)
subprocess.run([str(PY), "-m", "pip", "check"], check=True)
probe = (
    "import sys,torch; print(sys.executable,sys.version,torch.__version__,torch.version.cuda);"
    "assert torch.cuda.is_available(); print(torch.cuda.get_device_name(),"
    "torch.cuda.get_device_properties(0).total_memory)"
)
subprocess.run([str(PY), "-c", probe], check=True)
print("All model processes use:", PY, "; kernel imports no vendor package")

In [ ]:
# Use the actual process interpreter, not a kernel's different torch installation.
probe = (
    "import json,psutil,torch;"
    "print(json.dumps({'gpu':torch.cuda.get_device_name(),"
    "'vram_gib':torch.cuda.get_device_properties(0).total_memory/2**30,"
    "'host_available_gib':psutil.virtual_memory().available/2**30,"
    "'bf16':torch.cuda.is_bf16_supported()}))"
)
resource = json.loads(subprocess.check_output([str(PY), "-c", probe], text=True))
print(resource)
print("CPU offload requires >=64 GiB available host RAM as a conservative safety gate.")
print("This is not a GPU model whitelist or a guarantee of feasibility.")

In [ ]:
link = ROOT / "data/source_cache"
if not link.exists():
    link.symlink_to(CACHE, target_is_directory=True)
prepare_code = """
import json
from pathlib import Path
from scripts.prepare_monash_variants import prepare
from tsfm_crossover.data.common import stable_hash
from tsfm_crossover.data.pilot_data import prepare_one
root=Path.cwd()
pins=json.loads((root/'results/manifests/pilot/prepared_primary.json').read_text())
result=prepare('Traffic', root)
assert stable_hash(result)==stable_hash(pins['Traffic'])
ett=pins['ETTh1']
prepare_one('ETTh1', root, root/'data/source_cache',
            {s['url']:s['sha256'] for s in ett['source_files']})
print('Pinned Traffic and ETTh1 prepared; runtime rechecks fingerprints.')
"""
subprocess.run([str(PY), "-c", prepare_code], check=True)

In [ ]:
assert input("Type PROBE_TRAFFIC to run bounded memory probes: ") == "PROBE_TRAFFIC"


def run_probe(dataset, profile):
    dest = OUT / f"{dataset}-{profile}.json"
    log = OUT / f"{dataset}-{profile}.log"
    if not dest.exists():
        with log.open("w") as handle:
            done = subprocess.run(
                [
                    str(PY),
                    "-m",
                    "tsfm_crossover.experiments.traffic_memory",
                    "--dataset",
                    dataset,
                    "--profile",
                    profile,
                    "--expected-commit",
                    COMMIT,
                    "--output",
                    str(dest),
                ],
                stdout=handle,
                stderr=subprocess.STDOUT,
            )
        if done.returncode != 0:
            raise RuntimeError(f"Probe process failed; preserve and inspect {log.name}")
    result = json.loads(dest.read_text())
    print(dataset, profile, result["status"], result.get("reason"), result.get("error"))
    if result["status"] == "failed" and result.get("error", {}).get("category") != "out_of_memory":
        raise RuntimeError("Non-memory failure: stop and inspect; no automatic precision fallback.")
    if result["status"] == "running":
        raise RuntimeError(
            "Previous process interrupted; retain evidence and use a new run directory."
        )
    return result


reference = run_probe("ETTh1", "fp32")
assert reference["status"] == "passed", "Stop: reference environment or model check failed."
success = False
if resource["vram_gib"] >= 70:
    result = run_probe("Traffic", "fp32")
    success = result["status"] == "passed"
if not success:
    result = run_probe("ETTh1", "fp32_cpu_saved")
    if result["status"] == "passed":
        result = run_probe("Traffic", "fp32_cpu_saved")
        success = result["status"] == "passed"
if not success and resource["bf16"]:
    result = run_probe("ETTh1", "bf16")
    if result["status"] == "passed":
        result = run_probe("Traffic", "bf16")
        success = result["status"] == "passed"
if not success and resource["bf16"]:
    result = run_probe("ETTh1", "bf16_cpu_saved")
    if result["status"] == "passed":
        result = run_probe("Traffic", "bf16_cpu_saved")
        success = result["status"] == "passed"
print("Candidate smoke passed:", success)
print(
    "Not automatic primary inclusion or precision-policy approval. Return results even on failure."
)

In [ ]:
from google.colab import files

archive = Path("/content/traffic-memory-probes.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob("*.json"):
        assert p.stat().st_size < 2_000_000
        z.write(p, p.name)
files.download(str(archive))
print("Drive holds JSON and diagnostic logs; ZIP contains no weights or checkpoints.")